# Sample MERRA-2 at IMPROVE Sites
This is an example showing how to use GMAOpyobs utilities to read IMPROVE surface observations data, sample MERRA-2, and do a simple comparison.

This assumes that you have cloned the GMAOpyobs Github repository, installed it in a directory called `$AERODIR`, and have added the following path to your `$PYTHONPATH` environment variable:
`$AERODIR/install/lib/Python`

It also assumes you have a grads style control file for the MERRA-2 files. An example of such a file is in this directory called `inst3_3d_aer_Nv`, and it points to files in a directory called `MERRA2_all` also in this directory. 
The full path to `MERRA2_all` on discover is `/discover/nobackup/projects/gmao/merra2/data/products/MERRA2_all`.

To create the `MERRA2_all` directory in the current directory, create a symbolic link:
`ln -s /discover/nobackup/projects/gmao/merra2/data/products/MERRA2_all .`

The `inst3_3d_aer_Nv` MERRA-2 file collection contains 3-D aerosol mass mixing ratios on the native model vertical levels.

This example also requires GEOS aerosol optics files as inputs.  These files can be found at `https://portal.nccs.nasa.gov/datashare/iesa/aerosol/ExtData/` or on discover at `/discover/nobackup/projects/gmao/share/dasilva/fvInput/ExtData`

The IMROVE data was downloaded from `https://views.cira.colostate.edu/fed/DataFiles/` and can found on discover at `/discover/nobackup/projects/gmao/iesa/aerosol/data/AeroObs/IMPROVE`

## IMPROVE Station Sampling

In [1]:
# If you using discover JupyterHub, make sure to use the latest GEOSpyd kernel
# Add a kernel to your Jupyterhub by executing the following on discover:
# python3 -m ipykernel install --user --name GEOSpyD-24.11.3-3.13 --display-name "GEOSpyD 24.11.3 3.13"
# restart Jupyterhub and the new kernel should be available

# You only need to do these first three lines if you're using the discover JupyterHub
# replace the AERODIR path with your install location for GMAOpyobs
import sys
AERODIR = '/discover/nobackup/caturne4/GMAOpyobs'
sys.path.append(AERODIR+'/install/lib/Python')

from pyobs.sampler import STATION
from pyobs.improve import IMPROVE,SITE_MAP
from datetime import datetime
import xarray as xr
from pyobs.airNow import AIRNOW, AIRNOW_SITE_MAP
from datetime import timedelta
import numpy as np
import os

In [2]:
## Read the IMPROVE locations file
# site_path = '/discover/nobackup/projects/gmao/iesa/aerosol/data/AeroObs/IMPROVE/IMPROVE_locations.txt'
# sites = SITE_MAP(site_path)
# print('IMPROVE LOCATIONS')
# print(sites.df)
site_path = '/gpfsm/dnb34/caturne4/AirNow_API/Monitoring_Site_Locations_V2.dat'

#site_path = '/Users/pcastell/Documents/Asia-AQ/IMPROVE/improve/IMPROVE_locations.txt'
sites = AIRNOW_SITE_MAP(site_path)


In [7]:
## set the time range you want to consider
time_range = [datetime(2026,6,1),datetime(2026,6,15)]
date_range = f'{datetime.strftime(time_range[0], '%Y%m%d')}_{datetime.strftime(time_range[1], '%Y%m%d')}'
outFile = f'fp_airNow_sampled_{date_range}.nc4'
## Create a station object with aerosol data

fp_dir = '/discover/nobackup/projects/gmao/geos_fp_arch/f5430_fp/chem'
current_time = time_range[0]
end_time = time_range[1]
m2_data_files = []
while current_time <= end_time:
    # print(f'{int(current_time.hour):02d}')
    m2_data_files.append(fp_dir + f'/Y{current_time.year}/M{int(current_time.month):02d}/f5430_fp.inst3_3d_aer_Nv.{current_time.year}{int(current_time.month):02d}{int(current_time.day):02d}_{int(current_time.hour):02d}00z.nc4')
    current_time += timedelta(hours=3)

#m2data = ['inst3_3d_aer_Nv']  # MERRA-2 collection ctrl files
# m2_data_files = [fp_dir + '/Y2026/M05/f5430_fp.inst3_3d_aer_Nv.20260501_1200z.nc4', fp_dir + '/Y2026/M05/f5430_fp.inst3_3d_aer_Nv.20260502_1200z.nc4']

# print(m2_data_files)
if len(m2_data_files) > 1:
    m2data = xr.open_mfdataset(m2_data_files, combine='nested', concat_dim = 'time')
else:
    m2data = xr.open_dataset(m2d_data_files[0])
    
# print(m2data.data_vars)
stn = STATION(sites.df['AQSID'],sites.df['Longitude'],sites.df['Latitude'],m2data,time_range=time_range,verbose=True)
# print(stn.ds)

KeyboardInterrupt: 

In [8]:
## Sample the MERRA-2 dataset at the stations, and return an xarray dataset
# on Jupyterhub this will take a minute - stand up, get a cup of coffee
# on the command line, this takes ~7 seconds
if not os.path.exists(outFile):

# Variables I want to read (fp)
    du = ['DU001','DU002','DU003','DU004','DU005']
    ss = ['SS001','SS002','SS003','SS004','SS005']
    bc = ['BCPHILIC','BCPHOBIC']
    br = ['BRPHILIC', 'BRPHOBIC']
    oc = ['OCPHILIC','OCPHOBIC']
    su = ['SO4']
    ni = ['NI001','NI002','NI003']
    nh4 = ['NH4A']
    met = ['AIRDENS','DELP','PS','RH']
    
    Variables = met + du + ss + bc + oc + su + ni + nh4 + br
    
    stn_ds = stn.sample(Variables=Variables).compute()
    try:
        comp = dict(zlib=True)
        encoding = {var: comp for var in stn_ds.data_vars}
        stn_ds.to_netcdf(outFile,engine='netcdf4',encoding=encoding)
    except PermissionError:
        pass
else:
    outFile = f'fp_airNow_sampled_{date_range}.nc4'
    stn_ds = xr.open_dataset(outFile)
    
print('All Done :)')

All Done :)


# Calculate Modeled PM2.5 Comparable to IMPROVE Observations
Now that we have the 3-D aerosol mass mixing ratios, we would like to convert this to surface PM2.5 like what is measured by IMPROVE.

You will need the aerosol optics tables, and a yaml file describing which files to use.

The yaml file used here is called `m2_pm25.yaml` can be found in the  directory `GMAOpyobs/src/config`. 

It points to optics files located in the directory `ExtData`.

On discover, to create an `ExtData` directory in the current dirctory, create a symbolic link:

`ln -s /discover/nobackup/projects/gmao/share/dasilva/fvInput/ExtData .`


In [9]:
from pyobs.aop import G2GAOP

# set up some filenames
# this configuration file can be found in src/config
config = 'geos_fp_pm25.yaml'

In [10]:
# Create an optics object that links the model optics tables to the sampled aerosol profile data
optics = G2GAOP(stn_ds,config=config)

# ignore the phase matrix warning - not relevant here

In [11]:
# calculate PM2.5 per species
pm25 = {}

for spc in optics.mieTable:
    # print(spc)
    pm25[spc] = optics.getPM(Species=[spc],pmsize=2.5,aerodynamic=True)
    # add the dry mass, we will be comparing to that later
    pmdry = pm25[spc]['PM']*(1-pm25[spc]['FWATER'])
    attrs = {'long_name':'Dry Particulate Matter Mass', 'units':'microgram m-3'}
    pmdry.attrs.update(attrs)
    pm25[spc]['PMDRY'] = pmdry
    

spc = 'TOTAL'
pm25[spc] = optics.getPM(pmsize=2.5,aerodynamic=True)
# add the dry mass, we will be comparing to that later
pmdry = pm25[spc]['PM']*(1-pm25[spc]['FWATER'])
attrs = {'long_name':'Dry Particulate Matter Mass', 'units':'microgram m-3'}
pmdry.attrs.update(attrs)
pm25[spc]['PMDRY'] = pmdry
# print(pm25)

rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and rDry provided at 0% RH only; use v2.x.x tables and later
rLow and r

In [12]:
# optional: you can write this to a netcdf file
comp = dict(zlib=True)
for spc in pm25:
    outFile = f'fp_airNow_pm25_{spc}.nc4'
    print('writing ',outFile)
    encoding = {var: comp for var in pm25[spc].data_vars}
    pm25[spc].to_netcdf(outFile,engine='netcdf4',encoding=encoding)
# print(pm25)

writing  fp_airNow_pm25_DU.nc4
writing  fp_airNow_pm25_SS.nc4
writing  fp_airNow_pm25_OC.nc4
writing  fp_airNow_pm25_BC.nc4
writing  fp_airNow_pm25_BR.nc4
writing  fp_airNow_pm25_SU.nc4
writing  fp_airNow_pm25_NI.nc4
writing  fp_airNow_pm25_NH4.nc4
writing  fp_airNow_pm25_TOTAL.nc4


## Compare MERRA-2 PM2.5 to IMPROVE Observations

In [13]:
from pyobs.airNow import AIRNOW, AIRNOW_SITE_MAP
## Read the IMPROVE PM2.5 Obs

airNow_dir = '/gpfsm/dnb34/caturne4/AirNow_API/'
current_time = time_range[0]
end_time = time_range[1]
airnow_data_files = []
while current_time <= end_time:
    airnow_data_files.append(airNow_dir + f'HourlyData_{current_time.year}{int(current_time.month):02d}{int(current_time.day):02d}{int(current_time.hour):02d}.dat')
    current_time += timedelta(days=1)



site_path = '/gpfsm/dnb34/caturne4/AirNow_API/Monitoring_Site_Locations_V2.dat'
imp = AIRNOW(airnow_data_files,site_path=site_path)
sites = AIRNOW_SITE_MAP(site_path)
sites.df['tz_name'] = 'GMT'

In [ ]:
# Original IMPROVE code
# IMPROVE measures 24 hour average PM2.5 from midnight to midnight local time
# get the daily averages at each site
#####

from datetime import datetime,timedelta
import pytz

# time_range = [datetime(2026,4,5),datetime(2026,4,10)]
pm25_daily = {}
for spc in pm25:
    print(spc)
    tstart = time_range[0]
    tend = time_range[1]
    ds = pm25[spc]
    # print(ds)
    ds_daily = []
    while tstart <= tend:
        # print(tstart)
        ds_avg = []
        # print(sites.df)
        for site in sites.df['AQSID']:
            # print("AQSID sample:", sites.df['AQSID'].iloc[0], type(sites.df['AQSID'].iloc[0]))
            # print("ds station sample:", ds.station.values[0], type(ds.station.values[0]))
            # print('~~~')
            # print(site)
            if site not in ds.station.values:
                continue
            timezone_name = sites.df[sites.df['AQSID'] == site]['tz_name'].item()
            if timezone_name:
                local_timezone = pytz.timezone(timezone_name)
                time_aware_local = local_timezone.localize(tstart, is_dst=False)
                time_utc = time_aware_local.astimezone(pytz.utc).replace(tzinfo=None)
                ds_new = ds.sel(station=site).sel(time=slice(time_utc,time_utc+timedelta(hours=24))).mean(dim='time')
                ds_new = ds_new.assign_coords(time=[tstart])
                # print(ds_new)
                ds_avg.append(ds_new)
                # print(ds_avg)

        ds_avg = xr.concat(ds_avg,dim='station',coords="different",compat='equals',)
        ds_daily.append(ds_avg)

        tstart += timedelta(days=1)

    pm25_daily[spc] = xr.concat(ds_daily,dim='time',coords="different",compat='equals',data_vars='all')
# print(pm25_daily)

DU
SS
OC
BC
BR
SU
NI
NH4
TOTAL


In [ ]:
# convert IMPROVE dataframe to xarray dataset for easier comparisons
# df = merged.rename(columns={'SiteName':'station','Longitude':'lon','Latitude':'lat'})
import pandas as pd
df = imp.df.rename(columns={'SiteName':'station','Start_Time':'time','Longitude':'lon','Latitude':'lat', 'parameter name' : 'species'})

df = df.set_index(['Time','QSID','species'])

print(f"Original length: {len(df.index)}")
print(f"Unique index length: {len(df.index.unique())}")
# there  can be duplicate entries because there are multiple instruments at one site
# let's just keep the first measurement in the database
df_unique = df[~df.index.duplicated(keep='first')]
# Now convert the cleaned DataFrame to xarray
df_unique = df_unique.dropna()
df_unique['date_time'] = pd.to_datetime(df_unique['Date'] + ' ' + df_unique['time'], format='%m/%d/%y %H:%M')
# print(df_unique[df_unique['date_time'] == '2026-05-02T00:00:00.000000000']['value'].values.max())
df_unique = df_unique.reset_index()
df_unique = df_unique.set_index(['date_time', 'QSID', 'species'])



imp_ds = df_unique.to_xarray()


# imp_ds.sel(date_time = '2026-05-02T00:00:00.000000000')['value'].max().item()

In [ ]:
## Create a little plotting function
import matplotlib.pyplot as plt
def compare_imp_m2(imsubset,m2subset,species):

    obs_values = imsubset['value'].values.flatten()
    model_values = m2subset['PMDRY'].values.flatten()

    mask = ~np.isnan(obs_values) & ~np.isnan(model_values)

    obs_values = obs_values[mask]
    model_values = model_values[mask]

    
    im = plt.plot(obs_values, model_values,'o')

    # Get the range for the 1:1 line
    min_val = -2#min(imsubset['value'].min(), m2subset['PMDRY'].min())
    max_val = 30#max(imsubset['value'].max(), m2subset['PMDRY'].max())
    plt.xlim(min_val,max_val)
    plt.ylim(min_val,max_val)

    # Add 1:1 line
    line = plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 line')

    # Add labels and formatting
    plt.xlabel(f'AIRNOW {species} PM2.5')
    plt.ylabel(f'GEOS-FP {species} PM2.5')
    plt.title(f'AIRNOW vs GEOS-FP {species} Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()



In [ ]:
# Total PM2.5
m2spc = 'TOTAL'
imspc = 'PM2.5'
m2 = pm25_daily[m2spc]
im = imp_ds.sel(species=imspc)

im_dates = im.date_time.dt.floor('D')
m2_dates = m2.time.dt.floor('D')
valid_time_mask = np.isin(im_dates.values, m2_dates.values)
valid_times = im['date_time'].values[valid_time_mask]

# subset improve data for time period of model
# valid_times = im.date_time.values[np.isin(im.date_time.values, m2.time.values)]
# print(valid_times)
valid_stations = m2.station.values[np.isin(m2.station.values, im.QSID.values)]
# print(valid_stations)
imsubset = im.sel(
    date_time=valid_times,
    QSID=valid_stations
)
# repalce -999 with nan
imsubset = imsubset.where(imsubset != -999.)
# plot_df = imsubset['value'].to_dataframe().reset_index()
# plot_df = plot_df.dropna(subset=['value'])
# imsubset = imsubset.dropna(dim='date_time')
# subset model for stations in IMPROVE dataset
# and at the surface
m2subset = m2.sel(
    time=valid_times,
    station=valid_stations,
    lev=72
)
print('im values:\n',imsubset['value'].values)
print('m2 values:\n',m2subset['PMDRY'].values)
compare_imp_m2(imsubset,m2subset,'Total')

In [ ]:
# Black Carbon or elemental carbon
import numpy as np
m2spc = 'BC'
imspc = 'BCPHOBIC'
m2 = pm25_daily[m2spc]

im = imp_ds.sel(species=imspc)
print(im)
# subset improve data for time period of model
valid_times = im.date_time.values[np.isin(im.date_time.values, m2.time.values)]
valid_stations = m2.station.values[np.isin(m2.station.values, im.QSID.values)]

imsubset = im.sel(
    date_time=valid_times,
    QSID=valid_stations
)
# repalce -999 with nan
imsubset = imsubset.where(imsubset != -999.)

print('values:\n',imsubset)
# subset model for stations in IMPROVE dataset
# and at the surface
m2subset = m2.sel(
    time=valid_times,
    station=valid_stations,
    lev=72,
)

compare_imp_m2(imsubset,m2subset,'Black Carbon')